# 623 matched AMPM: stateful LSTM vs causal TCN

This is a parameter-paired architecture ablation on `623.xalancbmk_s-700B`, selected because historical AMPM is pollution-sensitive on this trace. AMPM, LSTM, and TCN use the same causal address stream, 64-page AMPM state, candidate bank, future-use labels, calibration rule, and keyed replayer. The neural policies can suppress AMPM candidates but cannot invent new candidates.

The TCN uses left-only padding with dilations `[1,2,4,8,16,32]` and a 127-event receptive field. Training chunks overlap by 126 context events, and overlap rows never contribute loss. The notebook runs three parameter-matched pairs: approximately 2.5K, 6K, and 13K parameters.

In [ ]:
import os, pathlib, shutil, subprocess, sys, tarfile, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU (A100 preferred)'
torch.set_float32_matmul_precision('high')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(torch.cuda.get_device_name(0), torch.__version__)

REPO = '/content/cache_arch'
PUBLIC_URL = 'https://github.com/Angelawoo572/cache_arch.git'
TOKEN = userdata.get('GITHUB_TOKEN')
assert TOKEN, 'Add GITHUB_TOKEN in Colab Secrets'
ASKPASS = '/content/cache_arch_git_askpass.sh'
pathlib.Path(ASKPASS).write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo x-access-token ;;\n  *) echo "$GITHUB_TOKEN" ;;\nesac\n')
os.chmod(ASKPASS, 0o700)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': ASKPASS, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': TOKEN})
try:
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', PUBLIC_URL, REPO], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only', 'origin', 'main'], check=True, env=git_env)
finally:
    pathlib.Path(ASKPASS).unlink(missing_ok=True)
print('Git:', subprocess.check_output(['git', '-C', REPO, 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_ID = '623_offline_lstm_tcn_ampm_seed7'
DRIVE_ROOT = f'/content/drive/MyDrive/cache_prefetch_623_ampm_temporal/{RUN_ID}'
INPUT_DIR = f'{DRIVE_ROOT}/colab_input'
OUTPUT_ROOT = f'{DRIVE_ROOT}/colab_output'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
INPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_input.tar.gz'
if not os.path.isfile(INPUT_ARCHIVE):
    from google.colab import files
    expected = f'{RUN_ID}.colab_input.tar.gz'
    uploaded = files.upload()
    assert expected in uploaded, f'Select {expected}; got {list(uploaded)}'
    pathlib.Path(INPUT_ARCHIVE).write_bytes(uploaded[expected])
with tarfile.open(INPUT_ARCHIVE, 'r:gz') as archive:
    archive.extractall(INPUT_DIR)
print('Input:', INPUT_ARCHIVE)
print('Persistent output:', OUTPUT_ROOT)


In [ ]:
import gzip, hashlib, json
TRACE = '623.xalancbmk_s-700B'
TRAIN = f'{INPUT_DIR}/{TRACE}.train_stream.csv.gz'
GUARD = f'{INPUT_DIR}/{TRACE}.guard_stream.csv.gz'
EVAL = f'{INPUT_DIR}/{TRACE}.eval_stream.csv.gz'
for path in (TRAIN, GUARD, EVAL):
    assert os.path.isfile(path), path
    digest = hashlib.sha256()
    with gzip.open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    print(path, digest.hexdigest())
SCRIPT = f'{REPO}/formal_NN_training/experiments/623_offline_lstm_tcn_ampm/python/train_and_offline_infer.py'
CONTRACT = f'{REPO}/formal_NN_training/experiments/623_offline_lstm_tcn_ampm/data/stream_contract.json'
assert os.path.isfile(SCRIPT), SCRIPT
print(json.dumps(json.loads(pathlib.Path(CONTRACT).read_text())['architecture_pairs'], indent=2))


In [ ]:
# All preprocessing and training run on local Colab disk; Drive stores only durable input/output.
LOCAL_INPUT = f'/content/{RUN_ID}_colab_input'
LOCAL_OUTPUT = f'/content/{RUN_ID}_colab_output'
for path in (LOCAL_INPUT, LOCAL_OUTPUT):
    if os.path.isdir(path):
        shutil.rmtree(path)
shutil.copytree(INPUT_DIR, LOCAL_INPUT)
LOCAL_TRAIN = f'{LOCAL_INPUT}/{TRACE}.train_stream.csv.gz'
LOCAL_GUARD = f'{LOCAL_INPUT}/{TRACE}.guard_stream.csv.gz'
LOCAL_EVAL = f'{LOCAL_INPUT}/{TRACE}.eval_stream.csv.gz'
MODEL_SPECS = [
    {'tag': 'lstm_h8',  'family': 'lstm', 'size': 8,  'pair_id': 'p2k'},
    {'tag': 'tcn_c10',  'family': 'tcn',  'size': 10, 'pair_id': 'p2k'},
    {'tag': 'lstm_h16', 'family': 'lstm', 'size': 16, 'pair_id': 'p6k'},
    {'tag': 'tcn_c16',  'family': 'tcn',  'size': 16, 'pair_id': 'p6k'},
    {'tag': 'lstm_h32', 'family': 'lstm', 'size': 32, 'pair_id': 'p13k'},
    {'tag': 'tcn_c24',  'family': 'tcn',  'size': 24, 'pair_id': 'p13k'},
]
SWEEP = []
for spec in MODEL_SPECS:
    local_out = f"{LOCAL_OUTPUT}/{spec['tag']}"
    drive_out = f"{OUTPUT_ROOT}/{spec['tag']}"
    if os.path.isdir(local_out): shutil.rmtree(local_out)
    if os.path.isdir(drive_out): shutil.rmtree(drive_out)
    cmd = [
        sys.executable, SCRIPT,
        '--train-stream', LOCAL_TRAIN,
        '--guard-stream', LOCAL_GUARD,
        '--eval-stream', LOCAL_EVAL,
        '--out-dir', local_out,
        '--model-family', spec['family'],
        '--model-size', str(spec['size']),
        '--pair-id', spec['pair_id'],
        '--device', 'cuda',
        '--seed', '7',
        '--epochs', '8',
        '--chunk-len', '1024',
        '--accumulate-chunks', '16',
    ]
    print('\nTraining', spec['tag'], 'command:', ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout: print(result.stdout, end='')
    if result.returncode != 0:
        if result.stderr: print(result.stderr, end='')
        raise RuntimeError(f"{spec['tag']} failed with exit code {result.returncode}")
    metadata = json.loads(pathlib.Path(f'{local_out}/run_metadata.json').read_text())
    assert metadata['model_tag'] == spec['tag']
    assert metadata['architecture_pair_id'] == spec['pair_id']
    assert metadata['matched_normal_prefetcher'] == 'ampm'
    assert metadata['model_does_not_use_pc'] is True
    assert metadata['causal_no_future_self_test'] == 'PASS'
    assert metadata['experiment_revision'] == 'architecture_ablation_v1'
    shutil.copytree(local_out, drive_out)
    SWEEP.append({key: metadata[key] for key in ('model_tag', 'model_family', 'model_size', 'architecture_pair_id', 'parameter_count', 'threshold', 'offline_ampm_entries', 'offline_nn_entries')})
pathlib.Path(f'{LOCAL_OUTPUT}/sweep_manifest.json').write_text(json.dumps({'trace': TRACE, 'revision': 'architecture_ablation_v1', 'points': SWEEP}, indent=2) + '\n')
shutil.copy2(f'{LOCAL_OUTPUT}/sweep_manifest.json', f'{OUTPUT_ROOT}/sweep_manifest.json')
print(json.dumps(SWEEP, indent=2))


In [ ]:
required = ['offline_ampm.replay.csv', 'offline_nn.replay.csv', 'model.pt', 'run_metadata.json', 'policy_sweep.csv']
for spec in MODEL_SPECS:
    out_dir = f"{OUTPUT_ROOT}/{spec['tag']}"
    assert all(os.path.isfile(f'{out_dir}/{name}') for name in required), out_dir
OUTPUT_ARCHIVE = f'{DRIVE_ROOT}/{RUN_ID}.colab_output.tar.gz'
LOCAL_ARCHIVE = f'/content/{RUN_ID}.colab_output.tar.gz'
with tarfile.open(LOCAL_ARCHIVE, 'w:gz') as archive:
    for item in pathlib.Path(LOCAL_OUTPUT).iterdir():
        archive.add(item, arcname=item.name)
shutil.copy2(LOCAL_ARCHIVE, OUTPUT_ARCHIVE)
print('DONE:', OUTPUT_ARCHIVE, os.path.getsize(OUTPUT_ARCHIVE), 'bytes')


After copying the output archive back to the server, run `STAGE=replay`. The analyzer will emit both the baseline-compatible metrics (`l2_load_miss_rate`, `selected_accuracy`, `coverage_vs_no_pref_l2_miss`, `timeliness`) and the capped balanced parity index. `live_spp_context` is presentation context only and is never treated as a matched neural comparator.